# Train the distillation teachers on PKU-Market-PCB (GPU)

The PKU twin of `02_train_teachers.ipynb`. Same ResNet-18 teacher, same schedule,
same seeds -- only the dataset differs, so a PKU teacher and a DeepPCB teacher are
comparable by construction.

This has to run on a GPU. Measured on this project's CPU box, the teacher costs about
**1.6 hours per epoch** on the PKU tile set (3,827 train crops, 4.8x DeepPCB's 800),
which is roughly two days per seed. On a T4 it is minutes.

**Before running:** Runtime -> Change runtime type -> **GPU**.

Outputs go to `runs_pku/teacher_s{0,1,2}/best.pt`, which is what
`training/run_matrix.py --set kd --out-dir runs_pku` expects.


## 1. Setup


In [ ]:
import os
import subprocess
import sys

import torch

BRANCH = "pku-dataset"
REPO = "https://github.com/vsubacius/PCBerrorDetection.git"

if not os.path.isdir("PCBerrorDetection"):
    subprocess.run(["git", "clone", "--branch", BRANCH, "--depth", "1", REPO], check=True)
os.chdir(os.path.abspath("PCBerrorDetection"))

print("cwd  ", os.getcwd())
print("torch", torch.__version__)
print("GPU  ", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE")
if not torch.cuda.is_available():
    raise SystemExit("Set Runtime -> Change runtime type -> GPU before running this.")


## 2. The PKU tile dataset

Two ways to get it, in order of preference:

1. **Upload the built `.npz`** (79 MB) to Drive and point `NPZ` at it. The file is
   `data/tiles_pku/pku_tiles_t160_s120.npz` on the machine that ran the conversion.
2. **Rebuild it here.** That clones the 3.7 GB PKU mirror and re-runs the converter
   (693 photographs -> 5,267 crops), which takes roughly 20-30 minutes on Colab.

Rebuilding is deterministic -- same crops, same folds -- so either route gives the
same dataset.


In [ ]:
import pathlib
import subprocess
import sys

NPZ = pathlib.Path("data/tiles_pku/pku_tiles_t160_s120.npz")
DRIVE_NPZ = None   # e.g. "/content/drive/MyDrive/pcb/pku_tiles_t160_s120.npz"

if DRIVE_NPZ and not NPZ.exists():
    from google.colab import drive
    drive.mount("/content/drive")
    NPZ.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(["cp", DRIVE_NPZ, str(NPZ)], check=True)

if not NPZ.exists():
    subprocess.run([sys.executable, "data/pku.py", "--download"], check=True)
    subprocess.run([sys.executable, "data/pku.py", "--convert"], check=True)
    subprocess.run([sys.executable, "data/build_tiles.py", "--dataset", "pku"], check=True)

print("tile dataset:", NPZ, f"{NPZ.stat().st_size / 1e6:.0f} MB")


## 3. Train the teachers

Input mode **C** (test + XOR), frozen from the DeepPCB input ablation and kept here so
the two datasets' teachers differ in one thing only.

`runs_pku/` keeps these apart from the DeepPCB teachers in `runs/`. Mixing them would
quietly distil a PKU student from a DeepPCB teacher.


In [ ]:
import sys
import time

sys.path.insert(0, "training")
from train import TrainConfig, train

SEEDS = (0, 1, 2)
REFERENCE_MODE = "C"
OUT_DIR = "runs_pku"

for seed in SEEDS:
    config = TrainConfig(
        run_id=f"teacher_s{seed}",
        model="teacher",
        representation=REFERENCE_MODE,
        loss_kind="plain",
        seed=seed,
        npz=str(NPZ),
        out_dir=OUT_DIR,
        device="cuda",
        num_workers=2,
    )
    if (config.run_dir / "metrics.json").exists():
        print(f"skip {config.run_id}: already finished")
        continue
    started = time.time()
    train(config)
    print(f"{config.run_id} done in {time.time() - started:.0f}s")


## 4. Save the checkpoints

Teacher weights are ~45 MB each and are **not** committed (`.gitignore` excludes
`runs/teacher*/*.pt`). Copy them to Drive, then download them to the machine that runs
the KD sweep and put them back at `runs_pku/teacher_s{seed}/best.pt`.


In [ ]:
import shutil

DRIVE_OUT = None   # e.g. "/content/drive/MyDrive/pcb/runs_pku"

if DRIVE_OUT:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    for seed in SEEDS:
        src = pathlib.Path(f"runs_pku/teacher_s{seed}")
        dst = pathlib.Path(DRIVE_OUT) / f"teacher_s{seed}"
        dst.mkdir(parents=True, exist_ok=True)
        for name in ("best.pt", "config.json", "metrics.json"):
            if (src / name).exists():
                shutil.copy2(src / name, dst / name)
        print("copied", src, "->", dst)
else:
    print("set DRIVE_OUT to copy the checkpoints out before the runtime recycles")


## 5. Then, back on the PC

```bash
python training/run_matrix.py --set kd \
    --out-dir runs_pku --npz data/tiles_pku/pku_tiles_t160_s120.npz
```

which runs Hinton KD and defect-aware KD against these teachers, one pair per seed.
